# 13. LCEL RAG 체인 — Retriever · Prompt · LLM · Parser
> Day 3 · 16H · 소요 약 50분

## 학습 목표

- LangChain `Chroma` 벡터스토어 + `OpenAIEmbeddings` 로 **retriever** 를 준비한다.
- LCEL 로 `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | parser` **표준 RAG 체인**을 조립한다.
- 같은 체인에 **스트리밍**(`stream`) 과 **fallback**(`with_fallbacks`) 을 적용한다.
- `MessagesPlaceholder` 로 **대화 히스토리**를 RAG 체인에 통합한다.

> **DB 미사용.** 이 노트북은 Neon 에 붙지 않고 작은 병원 안내 문서 코퍼스(in-notebook) 만으로 Chroma 를 채웁니다. 퍼시스트 경로는 `./lc_chroma` — 04/05/06/10/11 의 경로와 분리합니다.


In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-core langchain-chroma chromadb pydantic

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# 임베딩 + LLM 호출 모두 OpenAI 키 하나로 처리.
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

## 1. 작은 문서 코퍼스 + Chroma 벡터스토어

병원 안내용 짧은 문서 8 개를 만들어 `Chroma.from_documents(...)` 로 벡터화합니다. 퍼시스트 경로는 `./lc_chroma` 로 분리해 다른 노트북(04 LlamaIndex, 10 Vanna) 과 충돌하지 않게 합니다.


In [ ]:
# 작은 코퍼스로 Chroma 를 채워 한 노트북 안에서 RAG 흐름 전체를 시연합니다.
# LlamaIndex 의 Chroma 통합(05번)과 달리, 여기서는 LangChain 의 langchain-chroma 패키지를 사용합니다.
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536-차원, 학습용으로 비용 가성비 좋음
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)            # 답변 생성용

# Document(page_content=..., metadata={...}) — LangChain 의 표준 문서 객체.
# metadata 는 검색 결과를 분류·필터링할 때 활용 (예: dept="내과").
hospital_documents = [
    Document(page_content="내과에는 김철수(심장내과), 이영희(호흡기내과), 신민아(소화기내과) 전문의가 있습니다.", metadata={"dept": "내과"}),
    Document(page_content="외과에는 박민수(일반외과), 정수진(흉부외과), 권혁준(혈관외과)이 근무합니다.", metadata={"dept": "외과"}),
    Document(page_content="진료 시간은 평일 09:00-18:00, 토요일 09:00-13:00 입니다. 점심시간은 12:30-13:30.", metadata={"type": "schedule"}),
    Document(page_content="응급실은 24시간 운영됩니다. 야간에는 내과, 외과 당직의가 상주합니다.", metadata={"type": "emergency"}),
    Document(page_content="입원 병실 가격: 1인실 250,000원/일, 2인실 150,000원/일, 4인실 80,000원/일.", metadata={"type": "admission"}),
    Document(page_content="소아과에는 최동현, 강미래, 문서영 전문의가 있으며 소아청소년 질환 전반을 진료합니다.", metadata={"dept": "소아과"}),
    Document(page_content="정형외과에는 윤성호(척추외과), 한지은(관절외과) 전문의가 근무합니다.", metadata={"dept": "정형외과"}),
    Document(page_content="외래 환자는 주차 3시간 무료, 이후 30분당 1,000원. 입원 환자 보호자는 1일 5,000원 정액제.", metadata={"type": "parking"}),
]

# Chroma.from_documents — 문서 리스트를 받아 자동으로 임베딩 → 컬렉션에 저장까지.
# persist_directory 를 주면 디스크에 영속됩니다 (다른 노트북과 분리하기 위해 별도 폴더 사용).
vectorstore = Chroma.from_documents(
    hospital_documents,
    embeddings,
    collection_name="hospital_rag",
    persist_directory="./lc_chroma",
)
# .as_retriever() 로 Runnable 인터페이스를 가진 retriever 객체 생성 → LCEL 파이프에 바로 연결 가능.
# search_kwargs={"k": 3} — 가장 가까운 청크 3개만 가져옴 (k 가 크면 컨텍스트가 늘어 토큰 비용 증가).
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Vectorstore ready: {len(hospital_documents)} docs, top-k=3")

## 2. LCEL 표준 RAG 체인 조립

핵심 패턴 한 줄:

```python
{"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
```

- `"context"` 자리 — 원본 질문이 `retriever` 로 흘러가 `Document` 리스트가 되고, `format_docs` 로 하나의 텍스트로 합쳐집니다.
- `"question"` 자리 — `RunnablePassthrough` 가 원본 질문을 그대로 보존합니다.
- 최종적으로 `rag_prompt` 의 `{context}` 와 `{question}` 이 동시에 채워져 LLM 에 전달됩니다.


In [ ]:
# LCEL 표준 RAG 체인 — 이 패턴 한 줄이 거의 모든 RAG 튜토리얼의 출발점입니다.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """검색된 Document 리스트를 한 덩어리 텍스트로 합친다 — 프롬프트의 {context} 자리에 들어갈 형식."""
    # 각 문서의 page_content 만 뽑아 두 줄 띄움(\n\n) 으로 구분.
    return "\n\n".join(doc.page_content for doc in docs)


# 답변 형식을 강제하는 시스템 메시지 — "컨텍스트 외 정보는 만들지 말라" 가 핵심 규칙.
rag_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트를 바탕으로 질문에 한국어로 답변하세요.\n"
    "컨텍스트에 없는 내용은 \"해당 정보가 없습니다\" 라고 답변하세요.\n\n"
    "## 컨텍스트\n{context}\n\n"
    "## 질문\n{question}\n\n"
    "## 답변"
)

# RAG 체인의 핵심 패턴:
#   {"context": retriever | format_docs, "question": RunnablePassthrough()}
# 이 dict 는 LangChain 에서 자동으로 RunnableParallel 로 변환됩니다 — 즉 두 분기가 병렬 실행:
#   ① 입력 질문 → retriever → 문서 리스트 → format_docs → context 문자열
#   ② 입력 질문 → RunnablePassthrough() → 그대로 question 으로 흐름
# 두 결과가 dict 로 합쳐져 다음 단계인 rag_prompt 의 {context}, {question} 변수를 채웁니다.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("[Q] 내과에 어떤 의사가 있나요?")
print(rag_chain.invoke("내과에 어떤 의사가 있나요?"))
print("\n---\n")
print("[Q] 주차 요금이 어떻게 되나요?")
print(rag_chain.invoke("주차 요금이 어떻게 되나요?"))

## 3. 스트리밍 출력

같은 체인을 그대로 `.stream(...)` 로 호출하면 토큰 단위 스트림이 리턴됩니다. UI 를 만들 때 그대로 `yield` 로 연결할 수 있습니다.


In [ ]:
print("[stream] 입원 1 인실 비용은? ", end="")
for chunk in rag_chain.stream("입원 1인실 비용은 얼마인가요?"):
    print(chunk, end="", flush=True)
print()


## 4. Fallback — 고가 모델이 실패하면 저가 모델로

`primary_llm.with_fallbacks([fallback_llm])` 는 1 차 호출이 실패했을 때 자동으로 2 차 모델을 호출합니다. 대시보드 앞단에서 **가용성 확보**가 필요할 때 유용합니다.


In [ ]:
# Fallback — 1차 LLM 호출이 실패(레이트 리밋·타임아웃 등) 했을 때 자동으로 2차 모델로 재시도.
# 가용성이 중요한 운영 환경에서 자주 쓰이는 안전망 패턴입니다.
primary_llm  = ChatOpenAI(model="gpt-4o", temperature=0)        # 1차: 큰 모델
fallback_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # 2차: 작은 모델

# .with_fallbacks([…]) 로 감싸 주면, primary 가 예외를 던져도 LangChain 이 fallback_llm 을 자동 호출.
rag_chain_robust = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | primary_llm.with_fallbacks([fallback_llm])
    | StrOutputParser()
)

print(rag_chain_robust.invoke("응급실 운영 시간은 어떻게 되나요?"))

## 5. 구조화 출력 RAG — Pydantic 으로 받기

자유 텍스트 답변 대신 **답변 + 근거 문서 수 + 신뢰도** 같은 구조를 받고 싶을 때, 12 번에서 배운 `with_structured_output` 을 RAG 체인에도 똑같이 끼워 넣을 수 있습니다.


In [ ]:
from pydantic import BaseModel, Field


class HospitalAnswer(BaseModel):
    """RAG 답변 + 메타정보"""

    answer: str = Field(description="사용자에게 제공할 한국어 답변")
    grounded: bool = Field(description="컨텍스트만으로 답변 가능한지 여부")
    category: str = Field(description="답변 카테고리: doctor, schedule, emergency, admission, parking, other")


structured_llm = llm.with_structured_output(HospitalAnswer)

structured_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | structured_llm
)

result = structured_rag_chain.invoke("정형외과에 어떤 의사가 있어?")
print("answer   :", result.answer)
print("grounded :", result.grounded)
print("category :", result.category)


## 6. 멀티턴 RAG — 대화 히스토리 통합

이전 대화(`chat_history`) 를 프롬프트에 함께 실어 주면 "그 중에서 심장 전문의는?" 같은 **지시 대명사**를 해석할 수 있습니다. `MessagesPlaceholder` 가 히스토리 삽입 위치를 지정합니다.


In [ ]:
# 멀티턴 RAG — 대화 히스토리를 함께 프롬프트에 실어 "그 중에 ..." 같은 지시 표현을 해석.
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# from_messages 는 system / human / placeholder 를 명시적으로 나눠 작성합니다.
# MessagesPlaceholder(variable_name="chat_history") = 이 자리에 과거 대화 메시지 리스트가 그대로 삽입됩니다.
history_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 병원 안내 도우미입니다. 아래 컨텍스트만을 근거로 한국어로 답변하세요.\n\n컨텍스트:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# 입력은 dict({"question":..., "chat_history":[...]}) 형태이므로,
# lambda 로 dict 의 어느 키를 어디로 흘려보낼지 명시적으로 라우팅한다.
history_rag_chain = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs,  # 질문만 retriever 로
        "chat_history": lambda x: x["chat_history"],                    # 히스토리는 placeholder 로
        "question": lambda x: x["question"],                            # 질문 원문은 human 메시지로
    }
    | history_rag_prompt
    | llm
    | StrOutputParser()
)

# chat_history 는 HumanMessage / AIMessage 의 리스트. 매 턴 후 사용자/AI 메시지를 차례로 append.
chat_history = []

q1 = "내과에 어떤 의사가 있어?"
a1 = history_rag_chain.invoke({"question": q1, "chat_history": chat_history})
print(f"Q1: {q1}\nA1: {a1}\n")
chat_history.extend([HumanMessage(content=q1), AIMessage(content=a1)])

q2 = "그 중에 심장 전문의는 누구야?"   # ← "그 중" = 직전 답변에 등장한 인물 → 히스토리 필요
a2 = history_rag_chain.invoke({"question": q2, "chat_history": chat_history})
print(f"Q2: {q2}\nA2: {a2}\n")
chat_history.extend([HumanMessage(content=q2), AIMessage(content=a2)])

q3 = "그 의사는 야간에도 당직을 서?"
a3 = history_rag_chain.invoke({"question": q3, "chat_history": chat_history})
print(f"Q3: {q3}\nA3: {a3}")

## 정리 — RAG 체인의 해부도

```
 질문
  │
  ├─► retriever ──► Document[] ──► format_docs ──► context 문자열
  │                                                        ▼
  └─────────────────── question 문자열 ─────────────► rag_prompt
                                                           │
                                                           ▼
                                                          LLM
                                                           │
                                                           ▼
                                                       StrOutputParser
```

- `format_docs` 는 단순해 보이지만 **검색 결과를 프롬프트로 옮기는 브릿지** — 필요에 따라 "상위 N 개만", "메타데이터 포함" 등으로 확장합니다.
- `with_fallbacks`, `with_structured_output`, `MessagesPlaceholder` 는 이 3 단 체인에 **Runnable 데코레이터** 처럼 덧붙이기만 하면 됩니다.
- 이 패턴은 14~15 번의 Advanced RAG (HyDE · Multi-Query · Hybrid · Re-rank) 와 17 번(LangGraph SQL 에이전트) 에서 반복 재사용됩니다.


## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. batch로 3개 질문 동시 실행
`batch` 메서드를 사용하면 여러 질문을 동시에 처리할 수 있습니다.

아래 3개 질문에 대해 (1) 순차 실행(`for` + `invoke`)과 (2) 배치 실행(`rag_chain.batch(...)`)의 시간을 각각 측정하고, 속도 향상 배수를 출력하세요.

1. 내과에 어떤 의사가 있나요?
2. 입원 병실 가격을 알려주세요
3. 주차 요금이 어떻게 되나요?

_힌트: `time.time()` 으로 시작·종료 시각을 찍어 차이를 구하고, 두 시간의 비율로 속도 향상을 계산합니다. 답변은 `[:100]` 로 잘라 미리보기만 출력하세요._


In [ ]:
# ============================================================
# 실습 과제 — batch로 3개 질문 동시 실행
# ============================================================

# 실습 1: batch 로 3개 질문 동시 실행
# TODO: 순차 실행(for + invoke)과 배치 실행(rag_chain.batch(...))의 시간을 time.time() 으로 측정하고 속도 향상 배수를 출력하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

**`14_advanced_rag_query.ipynb`** 에서 "질문 그대로는 검색이 안 되는" 상황을 해결합니다 — **HyDE**(가상 문서 생성), **Multi-Query**(질문 다각화), **Query Decomposition**(복잡 질문을 하위 질문으로 분해) 을 동일 질문셋에 적용해 **검색 품질이 얼마나 달라지는지**를 정량 비교합니다.
